# LeetCode 374 — Guess Number Higher or Lower bằng Python

**Link đề:** https://leetcode.com/problems/guess-number-higher-or-lower/
**Độ khó:** Easy · **Chủ đề:** Binary Search, Interactive

```python
# The guess API is already defined for you.
# @param num, your guess
# @return -1 if num is higher than the picked number
#          1 if num is lower than the picked number
#          otherwise return 0
# def guess(num: int) -> int:

class Solution:
    def guessNumber(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu bài **tương tác (interactive)**: ta không thấy đáp án, chỉ hỏi được qua một **API** `guess(num)`.
2. Tự dựng lại API `guess` trong notebook để chạy độc lập (kèm bộ đếm số lần hỏi).
3. Đi từ đoán lần lượt `O(n)` → **tìm nhị phân (binary search)** `O(log n)`; biết thêm **ternary search** và `bisect` với `key`.
4. Trace `lo`, `hi`, `mid` và phản hồi `-1/0/1` từng bước; chứng minh tính đúng; đếm số lần gọi API.
5. Nắm chặt quy ước dấu dễ nhầm: `-1` nghĩa là **đoán cao quá**.

## 1. Đề bài

Trò chơi đoán số: tôi chọn một số `pick` trong `[1, n]` (không đổi trong suốt trò chơi). Bạn đoán; mỗi lần sai tôi cho biết số của tôi **cao hơn** hay **thấp hơn** số bạn đoán.

Bạn gọi API có sẵn `guess(num)` trả về:
- `-1`: số bạn đoán **cao hơn** số tôi chọn (`num > pick`).
- `1`: số bạn đoán **thấp hơn** số tôi chọn (`num < pick`).
- `0`: đoán đúng (`num == pick`).

Trả về số tôi đã chọn.

| `n` | `pick` | Output |
|---:|---:|---:|
| 10 | 6 | 6 |
| 1 | 1 | 1 |
| 2 | 1 | 1 |

**Constraints:** `1 <= n <= 2^31 - 1`; `1 <= pick <= n`.

**Chú ý:** `pick` **không** được truyền vào hàm — ta chỉ biết nó qua `guess`. Trên LeetCode, `guess` là một **hàm toàn cục** đã được định nghĩa sẵn.

### 1.1. Đọc kỹ quy ước dấu (chỗ hay nhầm nhất)

Hãy coi `guess(num)` trả lời câu hỏi "**`pick` so với `num` thế nào?**":

| `guess(num)` | Nghĩa | Ta nên đi về phía |
|---:|---|---|
| `-1` | `pick < num` — đoán cao quá | **trái** (`hi = num - 1`) |
| `1` | `pick > num` — đoán thấp quá | **phải** (`lo = num + 1`) |
| `0` | `pick == num` | dừng |

Mẹo nhớ: kết quả giống **dấu của `pick − num`**.

## 2. Nền tảng Python

### 2.1. Dựng lại API `guess` để notebook chạy độc lập

Ta viết hàm `set_pick(p)` để "giấu" số `p` vào biến toàn cục `_PICK`, và một hàm `guess(num)` đọc `_PICK` để trả lời. Thêm biến `CALLS` đếm số lần gọi để so sánh hiệu quả các cách.

`global X` trong hàm cho phép **gán** vào biến toàn cục `X` (nếu không có `global`, phép gán sẽ tạo biến cục bộ mới).

In [1]:
_PICK = None
CALLS = 0

def set_pick(p: int) -> None:
    global _PICK, CALLS
    _PICK = p
    CALLS = 0

def guess(num: int) -> int:
    global CALLS
    CALLS += 1
    if num > _PICK:
        return -1
    if num < _PICK:
        return 1
    return 0

set_pick(6)
print("guess(9) =", guess(9), "(cao quá)")
print("guess(3) =", guess(3), "(thấp quá)")
print("guess(6) =", guess(6), "(đúng)")
print("Số lần đã gọi:", CALLS)

guess(9) = -1 (cao quá)
guess(3) = 1 (thấp quá)
guess(6) = 0 (đúng)
Số lần đã gọi: 3


### 2.2. Chia nguyên `//` và điểm giữa

`(lo + hi) // 2` là số nguyên ở giữa (làm tròn xuống). Ví dụ `(1 + 10) // 2 = 5`. Dạng an toàn tràn số ở C++/Java: `lo + (hi - lo) // 2` — trong Python hai cách cho cùng kết quả.

In [2]:
for lo, hi in [(1, 10), (6, 10), (6, 7), (1, 2**31 - 1)]:
    print(lo, hi, "->", (lo + hi) // 2, lo + (hi - lo) // 2)

1 10 -> 5 5
6 10 -> 8 8
6 7 -> 6 6
1 2147483647 -> 1073741824 1073741824


### 2.3. `log₂(n)` — vì sao binary search nhanh

Mỗi lần hỏi, binary search **loại một nửa** khoảng. Sau `k` lần, khoảng còn khoảng `n / 2^k`. Cần `2^k ≥ n` ⇒ `k ≈ log₂ n`. Với `n = 2^31 − 1`, chỉ cần tối đa **31** lần hỏi.

In [3]:
import math
for n in [10, 1000, 10**6, 2**31 - 1]:
    print(f"n={n:>10}: tối đa {math.floor(math.log2(n)) + 1} lần hỏi")

n=        10: tối đa 4 lần hỏi
n=      1000: tối đa 10 lần hỏi
n=   1000000: tối đa 20 lần hỏi
n=2147483647: tối đa 31 lần hỏi


## 3. Cách 1 — Trực giác: đoán lần lượt 1, 2, 3, ...

Cách ai cũng nghĩ ra: thử từng số cho tới khi `guess` trả `0`. Đúng, nhưng tệ nhất phải hỏi `n` lần — với `n ≈ 2·10^9` là quá chậm (Time Limit Exceeded).

In [4]:
def guess_linear(n: int) -> int:
    for num in range(1, n + 1):
        if guess(num) == 0:
            return num
    return -1          # không xảy ra nếu đề đúng

set_pick(6)
print("Đáp án:", guess_linear(10), "| số lần hỏi:", CALLS)
set_pick(999)
print("Đáp án:", guess_linear(1000), "| số lần hỏi:", CALLS)

Đáp án: 6 | số lần hỏi: 6
Đáp án: 999 | số lần hỏi: 999


### 3.1. Nút thắt

Cách 1 **bỏ phí thông tin**: `guess` cho biết cả **hướng** (`-1` hay `1`), nhưng ta chỉ dùng thông tin "đúng/sai". Khi biết "cao quá", ta có thể loại **mọi** số từ `num` trở lên chỉ với một câu hỏi.

**Insight:** hỏi ở **giữa** khoảng còn lại — dù câu trả lời là gì, ta loại được một nửa. Đây là **binary search**.

## 4. Cách 2 — Binary search (lời giải chính)

**Đoạn 1 — khoảng ứng viên:** `lo = 1`, `hi = n`. Bất biến: `pick` luôn nằm trong `[lo, hi]`.

**Đoạn 2 — hỏi ở giữa:** `mid = (lo + hi) // 2`, `res = guess(mid)`.

**Đoạn 3 — thu hẹp:**
- `res == 0` → trả `mid`.
- `res == -1` (cao quá) → `hi = mid - 1`.
- `res == 1` (thấp quá) → `lo = mid + 1`.

In [5]:
# Thử bằng tay MỘT bước của Đoạn 2-3 với n = 10, pick = 6
set_pick(6)
lo, hi = 1, 10
mid = (lo + hi) // 2
res = guess(mid)
print(f"mid={mid}, guess={res}")
if res == 1:
    lo = mid + 1
print("Khoảng mới:", [lo, hi], "-> đã loại 5 số 1..5 chỉ bằng 1 câu hỏi")

mid=5, guess=1
Khoảng mới: [6, 10] -> đã loại 5 số 1..5 chỉ bằng 1 câu hỏi


In [6]:
def guess_binary(n: int) -> int:
    lo, hi = 1, n
    while lo <= hi:
        mid = (lo + hi) // 2
        res = guess(mid)
        if res == 0:
            return mid
        if res == -1:          # mid > pick
            hi = mid - 1
        else:                  # mid < pick
            lo = mid + 1
    return -1

for n, p in [(10, 6), (1, 1), (2, 1), (1000, 999), (2**31 - 1, 1702766719)]:
    set_pick(p)
    print(f"n={n}, pick={p} -> {guess_binary(n)} | số lần hỏi: {CALLS}")

n=10, pick=6 -> 6 | số lần hỏi: 3
n=1, pick=1 -> 1 | số lần hỏi: 1
n=2, pick=1 -> 1 | số lần hỏi: 1
n=1000, pick=999 -> 999 | số lần hỏi: 9
n=2147483647, pick=1702766719 -> 1702766719 | số lần hỏi: 31


## 5. TRACE từng bước — `n = 10`, `pick = 6`

| Bước | `lo` | `hi` | `mid` | `guess(mid)` | Nghĩa | Cập nhật |
|---:|---:|---:|---:|---:|---|---|
| 1 | 1 | 10 | 5 | 1 | 5 thấp quá | `lo = 6` |
| 2 | 6 | 10 | 8 | -1 | 8 cao quá | `hi = 7` |
| 3 | 6 | 7 | 6 | 0 | đúng | trả **6** |

Cột `lo, hi` là khoảng **trước** khi hỏi; khoảng luôn chứa `6`.

In [7]:
def trace_guess(n: int, pick: int) -> int:
    set_pick(pick)
    lo, hi = 1, n
    step = 1
    print(f"n={n}, pick bí mật={pick}")
    while lo <= hi:
        mid = (lo + hi) // 2
        res = guess(mid)
        meaning = {0: "đúng!", -1: "cao quá", 1: "thấp quá"}[res]
        if res == 0:
            print(f"Bước {step}: [{lo}, {hi}] mid={mid} guess={res:>2} ({meaning}) -> trả {mid}")
            return mid
        if res == -1:
            hi = mid - 1
        else:
            lo = mid + 1
        print(f"Bước {step}: mid={mid} guess={res:>2} ({meaning}) -> khoảng mới [{lo}, {hi}]")
        step += 1
    return -1

_ = trace_guess(10, 6)
print()
_ = trace_guess(10, 10)       # trường hợp pick ở biên phải

n=10, pick bí mật=6
Bước 1: mid=5 guess= 1 (thấp quá) -> khoảng mới [6, 10]
Bước 2: mid=8 guess=-1 (cao quá) -> khoảng mới [6, 7]
Bước 3: [6, 7] mid=6 guess= 0 (đúng!) -> trả 6

n=10, pick bí mật=10
Bước 1: mid=5 guess= 1 (thấp quá) -> khoảng mới [6, 10]
Bước 2: mid=8 guess= 1 (thấp quá) -> khoảng mới [9, 10]
Bước 3: mid=9 guess= 1 (thấp quá) -> khoảng mới [10, 10]
Bước 4: [10, 10] mid=10 guess= 0 (đúng!) -> trả 10


Trong cell trên, `{0: "đúng!", -1: "cao quá", 1: "thấp quá"}[res]` là **tra dict ngay tại chỗ**: tạo dict rồi lấy giá trị theo khóa `res` — gọn hơn chuỗi `if/elif`.

## 6. Cách 3 — Ternary search (chia ba)

Một biến thể "tự nhiên": hỏi **hai** điểm `m1`, `m2` chia khoảng thành 3 phần, loại 2/3 khoảng mỗi vòng. Nghe có vẻ nhanh hơn, nhưng mỗi vòng tốn **tới 2 lần hỏi**, nên tổng số lần hỏi tệ nhất là `2·log₃ n ≈ 1.26·log₂ n` — **nhiều hơn** binary search. Ta code để tự kiểm chứng.

In [8]:
def guess_ternary(n: int) -> int:
    lo, hi = 1, n
    while lo <= hi:
        third = (hi - lo) // 3
        m1, m2 = lo + third, hi - third
        r1 = guess(m1)
        if r1 == 0:
            return m1
        r2 = guess(m2)
        if r2 == 0:
            return m2
        if r1 == -1:                 # pick < m1
            hi = m1 - 1
        elif r2 == 1:                # pick > m2
            lo = m2 + 1
        else:                        # m1 < pick < m2
            lo, hi = m1 + 1, m2 - 1
    return -1

n = 10**6
worst_bin = worst_ter = 0
for p in range(1, n + 1, 997):
    set_pick(p); guess_binary(n); worst_bin = max(worst_bin, CALLS)
    set_pick(p); guess_ternary(n); worst_ter = max(worst_ter, CALLS)
print(f"n={n}: số lần hỏi tệ nhất (mẫu) — binary: {worst_bin}, ternary: {worst_ter}")

n=1000000: số lần hỏi tệ nhất (mẫu) — binary: 20, ternary: 25


## 7. Cách 4 — Dùng `bisect` với `key` (Python 3.10+)

`bisect_left(a, x, lo, hi, key=f)` tìm vị trí đầu tiên `i` trong `range(lo, hi)` mà `f(a[i]) >= x`, bằng binary search. Ta không cần list thật: `range(1, n + 1)` hỗ trợ truy cập theo chỉ số mà **không tốn bộ nhớ**.

Định nghĩa `f(num) = -guess(num)`: với `num < pick` thì `f = -1`, `num == pick` thì `f = 0`, `num > pick` thì `f = 1` → dãy `f` **không giảm**. Số đầu tiên có `f(num) >= 0` chính là `pick`.

In [9]:
import sys
from bisect import bisect_left

print("Python", sys.version.split()[0])
r = range(1, 11)
print("range hỗ trợ chỉ số:", r[0], r[5], len(r))

def guess_bisect(n: int) -> int:
    nums = range(1, n + 1)
    i = bisect_left(nums, 0, key=lambda num: -guess(num))
    return nums[i]

set_pick(6)
print("Đáp án:", guess_bisect(10), "| số lần hỏi:", CALLS)
set_pick(1702766719)
print("Đáp án:", guess_bisect(2**31 - 1), "| số lần hỏi:", CALLS)

Python 3.13.18
range hỗ trợ chỉ số: 1 6 10
Đáp án: 6 | số lần hỏi: 3
Đáp án: 1702766719 | số lần hỏi: 31


**`lambda num: -guess(num)`** là hàm ẩn danh một dòng, tương đương:

```python
def f(num):
    return -guess(num)
```

Cách 4 gọn nhưng "giấu" logic; trong phỏng vấn nên viết binary search tay (Cách 2). Lưu ý `key=` cần Python ≥ 3.10.

## 8. Code nộp LeetCode

Trên LeetCode **không** định nghĩa `guess` — nó có sẵn. Ở notebook, `guess` là bản ta dựng ở mục 2.1.

In [10]:
class Solution:
    def guessNumber(self, n: int) -> int:
        lo, hi = 1, n
        while lo <= hi:
            mid = (lo + hi) // 2
            res = guess(mid)
            if res == 0:
                return mid
            if res < 0:
                hi = mid - 1
            else:
                lo = mid + 1
        return -1


sol = Solution()
for n, p in [(10, 6), (1, 1), (2, 1)]:
    set_pick(p)
    print(f"n={n}, pick={p} -> {sol.guessNumber(n)}")

n=10, pick=6 -> 6
n=1, pick=1 -> 1
n=2, pick=1 -> 1


## 9. Vì sao đúng?

**Invariant:** *`lo ≤ pick ≤ hi` ở đầu mỗi vòng lặp.*

- Ban đầu `[1, n]` chứa `pick` theo constraints.
- `guess(mid) = -1` ⇒ `pick < mid` ⇒ `pick ≤ mid − 1` ⇒ gán `hi = mid − 1` giữ invariant. Trường hợp `1` đối xứng.
- Khoảng giảm ít nhất 1 mỗi vòng (vì loại cả `mid`) ⇒ vòng lặp kết thúc.
- Vì khoảng luôn chứa `pick`, nó **không bao giờ rỗng** trước khi gặp `mid == pick` ⇒ hàm luôn trả về ở nhánh `res == 0`; dòng `return -1` chỉ để đủ cú pháp.

**Số lần hỏi:** mỗi vòng khoảng còn tối đa `⌊(hi − lo + 1) / 2⌋` ⇒ tối đa `⌊log₂ n⌋ + 1` lần hỏi.

In [11]:
# Kiểm chứng invariant bằng assert ngay trong vòng lặp, cho mọi pick với n = 50
def guess_binary_checked(n: int) -> int:
    lo, hi = 1, n
    while lo <= hi:
        assert lo <= _PICK <= hi, "invariant bị phá!"
        mid = (lo + hi) // 2
        res = guess(mid)
        if res == 0:
            return mid
        if res == -1:
            hi = mid - 1
        else:
            lo = mid + 1
    raise AssertionError("không thể tới đây nếu invariant đúng")

for p in range(1, 51):
    set_pick(p)
    assert guess_binary_checked(50) == p
print("Invariant lo <= pick <= hi đúng ở mọi vòng, với mọi pick trong [1, 50]")

Invariant lo <= pick <= hi đúng ở mọi vòng, với mọi pick trong [1, 50]


## 10. Độ phức tạp

| Cách | Số lần gọi `guess` (Time) | Space |
|---|---|---|
| 1. Đoán lần lượt | `O(n)` | `O(1)` |
| 2. Binary search | `O(log n)`, tối đa `⌊log₂ n⌋ + 1` = 31 với `n = 2^31 − 1` | `O(1)` |
| 3. Ternary search | `O(log n)` nhưng hằng số lớn hơn (~`2·log₃ n`) | `O(1)` |
| 4. `bisect` + `key` | `O(log n)` | `O(1)` (`range` không tạo list) |

## 11. Bộ test: mẫu đề, biên, đối chiếu, đếm số lần hỏi

In [12]:
methods = {
    "linear": guess_linear,
    "binary": guess_binary,
    "ternary": guess_ternary,
    "bisect": guess_bisect,
    "Solution": Solution().guessNumber,
}
cases = [(10, 6), (1, 1), (2, 1), (2, 2), (3, 2), (100, 1), (100, 100), (2**31 - 1, 1), (2**31 - 1, 2**31 - 1)]
for n, p in cases:
    for name, f in methods.items():
        if name == "linear" and n > 10**4:
            continue                       # cách tuyến tính quá chậm với n lớn
        set_pick(p)
        assert f(n) == p, (name, n, p)
        if name == "binary":
            assert CALLS <= n.bit_length(), ("hỏi quá nhiều", n, p, CALLS)
    print(f"n={n:>10}, pick={p:>10} PASS")

n=        10, pick=         6 PASS
n=         1, pick=         1 PASS
n=         2, pick=         1 PASS
n=         2, pick=         2 PASS
n=         3, pick=         2 PASS
n=       100, pick=         1 PASS
n=       100, pick=       100 PASS
n=2147483647, pick=         1 PASS
n=2147483647, pick=2147483647 PASS


In [13]:
# Duyệt hết mọi pick cho n nhỏ, và ngẫu nhiên có seed cho n lớn
for n in range(1, 60):
    for p in range(1, n + 1):
        for name in ["binary", "ternary", "bisect", "Solution"]:
            set_pick(p)
            assert methods[name](n) == p
import random
rng = random.Random(374)
max_calls = 0
for _ in range(2000):
    n = rng.randint(1, 2**31 - 1)
    p = rng.randint(1, n)
    set_pick(p)
    assert Solution().guessNumber(n) == p
    max_calls = max(max_calls, CALLS)
print("Tất cả PASS. Số lần hỏi nhiều nhất trong 2000 test ngẫu nhiên:", max_calls, "(<= 31)")

Tất cả PASS. Số lần hỏi nhiều nhất trong 2000 test ngẫu nhiên: 31 (<= 31)


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Đảo ngược ý nghĩa `-1` / `1`
Viết `if res == -1: lo = mid + 1` vì nghĩ "-1 là thấp hơn". Kết quả: đi sai hướng, khoảng loại mất `pick`.

### Lỗi 2 — `while lo < hi` với cập nhật `±1`
Khi khoảng còn đúng 1 số (`lo == hi`) thì vòng dừng mà chưa hỏi → trả `-1` sai (ví dụ `n = 1`).

### Lỗi 3 — Gọi `guess` nhiều lần cho cùng `mid`
`if guess(mid) == 0 ... elif guess(mid) == -1 ...` vẫn đúng nhưng **gấp đôi** số lần gọi API. Ở bài tương tác thật, mỗi lần gọi có thể tốn kém. Lưu vào biến `res`.

### Lỗi 4 — Dùng `/` thay `//`
`mid = (lo + hi) / 2` cho số thực (`5.5`), gọi `guess(5.5)` vô nghĩa và chỉ số không nguyên.

In [14]:
def wrong_direction(n, max_steps=100):
    lo, hi = 1, n
    steps = 0
    while lo <= hi and steps < max_steps:
        steps += 1
        mid = (lo + hi) // 2
        res = guess(mid)
        if res == 0:
            return mid
        if res == -1:
            lo = mid + 1          # SAI hướng
        else:
            hi = mid - 1
    return -1

def wrong_lt(n):
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        res = guess(mid)
        if res == 0:
            return mid
        if res == -1:
            hi = mid - 1
        else:
            lo = mid + 1
    return -1

def double_calls(n):
    lo, hi = 1, n
    while lo <= hi:
        mid = (lo + hi) // 2
        if guess(mid) == 0:
            return mid
        elif guess(mid) == -1:
            hi = mid - 1
        else:
            lo = mid + 1

set_pick(6); print("Lỗi 1 — đảo hướng, n=10, pick=6 ->", wrong_direction(10))
set_pick(1); print("Lỗi 2 — while lo < hi, n=1 ->", wrong_lt(1), "(đúng: 1)")
set_pick(1702766719); double_calls(2**31 - 1); c2 = CALLS
set_pick(1702766719); guess_binary(2**31 - 1); c1 = CALLS
print(f"Lỗi 3 — số lần gọi: gọi đôi {c2} vs lưu biến {c1}")
print("Lỗi 4 — (1 + 10) / 2 =", (1 + 10) / 2)

Lỗi 1 — đảo hướng, n=10, pick=6 -> -1
Lỗi 2 — while lo < hi, n=1 -> -1 (đúng: 1)
Lỗi 3 — số lần gọi: gọi đôi 61 vs lưu biến 31
Lỗi 4 — (1 + 10) / 2 = 5.5


## 13. Mẹo nhận dạng pattern

- Có **oracle trả lời hướng** ("cao hơn/thấp hơn", "đúng/sai từ một ngưỡng") trên miền **có thứ tự** → binary search.
- `n` tới `2^31 − 1` mà chỉ cho vài chục thao tác → bắt buộc `O(log n)`.
- Bài **Interactive**: luôn dựng một "giám khảo giả" (mock API) để tự test + **đếm số lần gọi**.
- Bài anh em: **278 First Bad Version** (`isBadVersion` trả True/False, tìm biên đầu tiên), **35 Search Insert Position**, **367 Valid Perfect Square**, **375 Guess Number Higher or Lower II** (DP — chi phí tối thiểu đảm bảo thắng).

## 14. Biến thể

### 14.1. First Bad Version (LeetCode 278) — tìm **biên** thay vì tìm chính xác

API `isBadVersion(v)` trả `False...False True...True`. Tìm `True` đầu tiên: template `while lo < hi`, `hi = mid` (giữ `mid` vì nó có thể là đáp án), `lo = mid + 1`.

In [15]:
def first_bad_version(n: int, is_bad) -> int:
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        if is_bad(mid):
            hi = mid          # mid có thể là bản lỗi đầu tiên -> giữ lại
        else:
            lo = mid + 1
    return lo

print(first_bad_version(5, lambda v: v >= 4))     # 4
print(first_bad_version(1, lambda v: v >= 1))     # 1

4
1


### 14.2. Đoán số khi phản hồi chỉ là "đúng/sai từ ngưỡng"

Nếu API chỉ trả `num >= pick` (True/False), ta vẫn tìm được `pick` bằng template "biên đầu tiên": `pick` là số nhỏ nhất có `num >= pick`.

In [16]:
def guess_with_ge(n: int, ge) -> int:
    return first_bad_version(n, ge)

set_pick(37)
print(guess_with_ge(100, lambda num: num >= _PICK))

37


## 15. Bài tập tự luyện (TODO)

**Bài 1 (tính tay).** `n = 100`, `pick = 73`. Lập bảng `lo, hi, mid, guess(mid)` của binary search. Cần mấy lần hỏi?

**Bài 2 (dự đoán output).** Với `n = 2^20`, số lần hỏi tệ nhất của binary search là bao nhiêu?

**Bài 3 (điền chỗ trống).** Hoàn thiện `exercise_guess`.

**Bài 4 (viết lại).** Viết binary search theo template **biên** (`while lo < hi`, `hi = mid`), dùng `guess(mid) <= 0` làm điều kiện "mid ≥ pick".

**Bài 5 (biến thể).** `guess_with_count(n)`: trả về cặp `(pick, số_lần_hỏi)` mà **không** dùng biến toàn cục `CALLS` (tự đếm).

**Bài 6 (mini challenge).** Đoán số trên **miền tùy ý**: API `guess(num)` vẫn trả `-1/0/1` như cũ nhưng `pick` nằm trong `[lo0, hi0]` bất kỳ (có thể âm). Viết `guess_range(lo0, hi0)`.

In [17]:
def exercise_guess(n: int) -> int:
    # TODO 1: lo, hi = 1, n
    # TODO 2: while lo <= hi: mid, res = guess(mid)
    # TODO 3: res == 0 -> return; res == -1 -> hi = mid - 1; res == 1 -> lo = mid + 1
    pass

# set_pick(73); assert exercise_guess(100) == 73

In [18]:
def exercise_boundary(n: int) -> int:
    # TODO: while lo < hi; nếu guess(mid) <= 0 thì hi = mid, ngược lại lo = mid + 1
    pass

# set_pick(73); assert exercise_boundary(100) == 73

In [19]:
def guess_with_count(n: int):
    # TODO: trả (pick, calls)
    pass

# set_pick(6); assert guess_with_count(10) == (6, 3)

In [20]:
def guess_range(lo0: int, hi0: int) -> int:
    # TODO
    pass

# set_pick(-17); assert guess_range(-100, 100) == -17

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | bắt đầu `[1, 100]` | mid đầu tiên là 50 | 50 → 75 → 62 → 68 → 71 → 73 |
| 2 | mỗi lần chia đôi | `log₂(2^20) = 20` | `⌊log₂ n⌋ + 1` |
| 3 | xem mục 4 | lưu `res = guess(mid)` | 3 nhánh |
| 4 | `mid ≥ pick` ⇔ `guess(mid) <= 0` | đúng → `hi = mid` | kết thúc trả `lo` |
| 5 | biến cục bộ `calls = 0` | tăng mỗi lần gọi `guess` | `return mid, calls` |
| 6 | chỉ đổi khoảng ban đầu | `//` làm tròn xuống cả với số âm | `lo, hi = lo0, hi0` |

## 17. Đáp án tham khảo

**Bài 1:** mid = 50 (thấp) → [51,100] mid 75 (cao) → [51,74] mid 62 (thấp) → [63,74] mid 68 (thấp) → [69,74] mid 71 (thấp) → [72,74] mid 73 đúng: **6 lần hỏi**.

**Bài 2:** `⌊log₂ 2^20⌋ + 1 = 21` lần (khoảng có `2^20` số: sau 20 lần còn 1 số, lần 21 hỏi chính nó).

In [21]:
def answer_guess(n):
    lo, hi = 1, n
    while lo <= hi:
        mid = (lo + hi) // 2
        res = guess(mid)
        if res == 0:
            return mid
        if res == -1:
            hi = mid - 1
        else:
            lo = mid + 1
    return -1

def answer_boundary(n):
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        if guess(mid) <= 0:        # mid >= pick
            hi = mid
        else:
            lo = mid + 1
    return lo

def answer_guess_with_count(n):
    lo, hi, calls = 1, n, 0
    while lo <= hi:
        mid = (lo + hi) // 2
        res = guess(mid)
        calls += 1
        if res == 0:
            return mid, calls
        if res == -1:
            hi = mid - 1
        else:
            lo = mid + 1
    return -1, calls

def answer_guess_range(lo0, hi0):
    lo, hi = lo0, hi0
    while lo <= hi:
        mid = (lo + hi) // 2
        res = guess(mid)
        if res == 0:
            return mid
        if res == -1:
            hi = mid - 1
        else:
            lo = mid + 1
    return None

_ = trace_guess(100, 73)
print("Bài 1: số lần hỏi =", CALLS)
worst = 0
n = 2**20
for p in [1, 2, n // 3, n // 2, n - 1, n]:
    set_pick(p); guess_binary(n); worst = max(worst, CALLS)
print("Bài 2: số lần hỏi lớn nhất trên các pick mẫu =", worst, "| công thức:", n.bit_length())
set_pick(73); assert answer_guess(100) == 73
set_pick(73); assert answer_boundary(100) == 73
set_pick(6); assert answer_guess_with_count(10) == (6, 3)
set_pick(-17); assert answer_guess_range(-100, 100) == -17
for p in range(1, 200):
    set_pick(p); assert answer_boundary(199) == p
print("Đáp án PASS toàn bộ!")

n=100, pick bí mật=73
Bước 1: mid=50 guess= 1 (thấp quá) -> khoảng mới [51, 100]
Bước 2: mid=75 guess=-1 (cao quá) -> khoảng mới [51, 74]
Bước 3: mid=62 guess= 1 (thấp quá) -> khoảng mới [63, 74]
Bước 4: mid=68 guess= 1 (thấp quá) -> khoảng mới [69, 74]
Bước 5: mid=71 guess= 1 (thấp quá) -> khoảng mới [72, 74]
Bước 6: [72, 74] mid=73 guess= 0 (đúng!) -> trả 73
Bài 1: số lần hỏi = 6
Bài 2: số lần hỏi lớn nhất trên các pick mẫu = 21 | công thức: 21
Đáp án PASS toàn bộ!


## 18. Template nhớ nhanh + Checklist submit

```python
class Solution:
    def guessNumber(self, n: int) -> int:
        lo, hi = 1, n
        while lo <= hi:
            mid = (lo + hi) // 2
            res = guess(mid)          # gọi 1 lần, lưu lại
            if res == 0:
                return mid
            if res < 0:               # -1: mid cao quá -> sang trái
                hi = mid - 1
            else:                     # 1: mid thấp quá -> sang phải
                lo = mid + 1
        return -1
```

**Checklist**
- [ ] Không tự định nghĩa lại `guess` trong code nộp (LeetCode đã có)?
- [ ] `-1` ⇒ `hi = mid - 1`; `1` ⇒ `lo = mid + 1` (đúng hướng)?
- [ ] `while lo <= hi` để không bỏ sót khoảng 1 phần tử (`n = 1`)?
- [ ] Mỗi vòng chỉ gọi `guess` **một** lần?
- [ ] Dùng `//`, đã test `pick = 1`, `pick = n`, `n = 2^31 − 1`?